# Data Scraping

This notebook contains code to scrap data on weather, UK bank holidays and covid times etc. The notebook also formats the data before modelling.

By the end of the notebook, the following variables should be collected, cleaned and joined:

Variable | Unit | Description | Source
---------|------|----------|----------
Estimated Actual Footfall | | Number of visitors in area for a given day. | [HUQ](https://huq.io/insights/footfall-data/)
Cos_weekday_num| |Cosine of the week day number | 
Sin_weekday_num| |Sinus of the week day number | 
Cos_month_num| |  Cosine of the month number| 
Sin_month_num| | Sinus of the month number| 
Cos_week_of_year| | Cosine of the week of year number| 
Sin_week_of_year| | Sinus of the week of year number | 
is_weekend | | Whether or not that day is a weekend (0= No, 1= Yes)|
bank_hol| |Whether or not that day is a bank holiday (0= No, 1= Yes)|[Kaggle](https://www.kaggle.com/datasets/shivd24coder/uk-national-holidays-dataset)
school_hol| |Whether or not that day is a school holiday (0= No, 1= Yes)|[Bradford Council](https://bradford.gov.uk/)
Covid times| |Whether or not that day was during covid restrictions (0= No, 1= Yes)|
Precipitation | mm | Sum of daily precipitation (including rain, showers and snowfall) | [Open Meteo API](https://open-meteo.com/en/docs/historical-weather-api?bounding_box=-90,-180,90,180&hourly=&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max)
Temperature | °C | Average daily air temperature at 2 meters above ground | [Open Meteo API](https://open-meteo.com/en/docs/historical-weather-api?bounding_box=-90,-180,90,180&hourly=&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max)
Wind Speed | km/h | Maximum wind speed on a day | [Open Meteo API](https://open-meteo.com/en/docs/historical-weather-api?bounding_box=-90,-180,90,180&hourly=&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max)
Daylight Duration | seconds | Number of seconds of daylight per day | [Open Meteo API](https://open-meteo.com/en/docs/historical-weather-api?bounding_box=-90,-180,90,180&hourly=&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max)

In [1]:
pip install matplotlib geopandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Import packages
import pandas as pd
import seaborn as sns
import matplotlib
import geopandas as gpd
import numpy as np

In [3]:
#Load footfall data (from '2- Footfall-Cleaning' notebook)
footfall = pd.read_csv(r"C:\Users\qxnq723\Desktop\Project 1\Coding\ML Analysis - Polygons with district\footfall_Mix_Clean.csv")
footfall.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27186 entries, 0 to 27185
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 27186 non-null  int64  
 1   datestamp                  27186 non-null  object 
 2   area                       27186 non-null  object 
 3   estimated_actual_footfall  27186 non-null  float64
 4   year                       27186 non-null  int64  
 5   month                      27186 non-null  int64  
 6   weekday                    27186 non-null  int64  
 7   week_of_year               27186 non-null  int64  
 8   Sin_weekday                27186 non-null  float64
 9   Cos_weekday                27186 non-null  float64
 10  Sin_monthday               27186 non-null  float64
 11  Cos_monthday               27186 non-null  float64
 12  Sin_week_of_year           27186 non-null  float64
 13  Cos_week_of_year           27186 non-null  flo

In [4]:
#Convert datestamp to datetime
footfall['datestamp'] = pd.to_datetime(footfall['datestamp'])

## Step 1: Weekends

In [5]:
#Create binary on whether day is a Saturday or Sunday
weekend = {5, 6}
#Add column in footfall data (1= weekend, 0= normal day)
footfall['is_weekend'] = footfall['weekday'].isin(weekend).astype(int)
#Check
footfall.head()

,Unnamed: 0,datestamp,area,estimated_actual_footfall,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_monthday,Cos_monthday,Sin_week_of_year,Cos_week_of_year,Sin_month,Cos_month,is_weekend
0,0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0
1,1,2019-01-01,Bradford - BID,54153.485,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0
2,2,2019-01-01,Bradford - Local Authority,530996.000,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0
3,3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0
4,4,2019-01-02,Bradford - BID,158891.385,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0


## Step 2: Bank Holidays
Bank holiday data was collected from Kaggle.

In [6]:
bank_hols = pd.read_csv(r"C:\Users\qxnq723\Desktop\Project 1\DATA\UK_holiday.csv")
bank_hols.head()

,title,date,notes,bunting
0,New Year’s Day,2018-01-01,NaN,True
1,Good Friday,2018-03-30,NaN,False
2,Easter Monday,2018-04-02,NaN,True
3,Early May bank holiday,2018-05-07,NaN,True
4,Spring bank holiday,2018-05-28,NaN,True


In [7]:
#Check date range
bank_hols['date'].unique()

array(['2018-01-01', '2018-03-30', '2018-04-02', '2018-05-07',
       '2018-05-28', '2018-08-27', '2018-12-25', '2018-12-26',
       '2019-01-01', '2019-04-19', '2019-04-22', '2019-05-06',
       '2019-05-27', '2019-08-26', '2019-12-25', '2019-12-26',
       '2020-01-01', '2020-04-10', '2020-04-13', '2020-05-08',
       '2020-05-25', '2020-08-31', '2020-12-25', '2020-12-28',
       '2021-01-01', '2021-04-02', '2021-04-05', '2021-05-03',
       '2021-05-31', '2021-08-30', '2021-12-27', '2021-12-28',
       '2022-01-03', '2022-04-15', '2022-04-18', '2022-05-02',
       '2022-06-02', '2022-06-03', '2022-08-29', '2022-09-19',
       '2022-12-26', '2022-12-27', '2023-01-02', '2023-04-07',
       '2023-04-10', '2023-05-01', '2023-05-08', '2023-05-29',
       '2023-08-28', '2023-12-25', '2023-12-26', '2024-01-01',
       '2024-03-29', '2024-04-01', '2024-05-06', '2024-05-27',
       '2024-08-26', '2024-12-25', '2024-12-26', '2025-01-01',
       '2025-04-18', '2025-04-21', '2025-05-05', '2025-

The data includes all bank holidays between 2019 and 2025, thus we can create a variable in the footfall data to indicate whether or not a day falls on a bank holiday, using the one hot encoding technique (0 is no, 1 is yes).

In [8]:
#Convert datestamp to datetime
bank_hols['date'] = pd.to_datetime(bank_hols['date'])
#Create set of bank holiday dates
holiday_set = set(bank_hols['date'])

In [9]:
#Add column in footfall data (1= bank hol, 0= normal day)
footfall['bank_hol'] = footfall['datestamp'].isin(holiday_set).astype(int)
#Check
footfall.head()

,Unnamed: 0,datestamp,area,estimated_actual_footfall,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_monthday,Cos_monthday,Sin_week_of_year,Cos_week_of_year,Sin_month,Cos_month,is_weekend,bank_hol
0,0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1
1,1,2019-01-01,Bradford - BID,54153.485,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1
2,2,2019-01-01,Bradford - Local Authority,530996.000,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1
3,3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0,0
4,4,2019-01-02,Bradford - BID,158891.385,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0,0


## Step 3: School Holidays
Data regarding the school holidays was manually scraped from calendars provided on the 'City of Bradford Metropolitan District Council' website. Thus, dates that fall within this school holiday periods will be assigned a 1 in the 'school_holiday' column.

In [10]:
#Create set of dates which were school holidays

school_hols_periods = [
    ('2019-01-01', '2019-01-04'),
    ('2019-02-18', '2019-02-22'),
    ('2019-04-15', '2019-04-26'),
    ('2019-05-27', '2019-05-31'),
    ('2019-07-22', '2019-07-31'),
    ('2019-08-01', '2019-08-31'),
    ('2019-10-28', '2019-10-30'),
    ('2019-05-27', '2019-05-31'),
    ('2019-12-23', '2019-12-31'),
    ('2020-01-01', '2020-01-03'),
    ('2020-02-17', '2020-02-19'),
    ('2020-04-06', '2020-04-17'),
    ('2020-05-25', '2020-05-29'),
    ('2020-07-21', '2020-09-02'),
    ('2020-10-26', '2020-10-28'),
    ('2020-12-21', '2021-01-01'),
    ('2021-02-15', '2021-02-17'),
    ('2021-03-29', '2021-04-09'),
    ('2021-06-01', '2021-06-04'),
    ('2021-07-26', '2021-09-03'),
    ('2021-10-25', '2021-10-27'),
    ('2021-12-20', '2022-01-03'),
    ('2022-02-21', '2022-02-23'),
    ('2022-04-11', '2022-04-22'),
    ('2022-05-31', '2022-06-03'),
    ('2022-07-27', '2022-09-02'),
    ('2022-10-24', '2022-10-26'),
    ('2022-12-19', '2023-01-02'),
    ('2023-02-13', '2023-02-15'),
    ('2023-04-03', '2023-04-14'),
    ('2023-05-29', '2023-06-02'),
    ('2023-07-26', '2023-09-01'),
    ('2023-10-23', '2023-10-25'),
    ('2023-12-18', '2024-01-01'),
    ('2024-02-12', '2024-02-14'),
    ('2024-03-25', '2024-04-05'),
    ('2024-05-27', '2024-05-31'),
    ('2024-07-24', '2024-08-30'),
    ('2024-10-28', '2024-11-01'),
    ('2024-12-23', '2025-01-03'),
    ('2025-02-17', '2025-02-21'),
    ('2025-04-07', '2025-04-18'),
    ('2025-05-26', '2025-05-30'),
    ('2025-07-23', '2025-08-29'),
    ('2025-10-27', '2025-10-31'),
    ('2025-12-22', '2026-01-02'),
]

school_hols_dates = pd.DatetimeIndex([])
for start, end in school_hols_periods:
    school_hols_dates = school_hols_dates.union(
        pd.date_range(start=start, end=end, freq='D'))
    
#Check
school_hols_dates

DatetimeIndex(['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04',
               '2019-02-18', '2019-02-19', '2019-02-20', '2019-02-21',
               '2019-02-22', '2019-04-15',
               ...
               '2025-12-24', '2025-12-25', '2025-12-26', '2025-12-27',
               '2025-12-28', '2025-12-29', '2025-12-30', '2025-12-31',
               '2026-01-01', '2026-01-02'],
              dtype='datetime64[ns]', length=541, freq=None)

In [11]:
#Add column in footfall data (1= school holiday, 0= normal day)
footfall['school_hol'] = footfall['datestamp'].isin(school_hols_dates).astype(int)
#Check
footfall.head()

,Unnamed: 0,datestamp,area,estimated_actual_footfall,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_monthday,Cos_monthday,Sin_week_of_year,Cos_week_of_year,Sin_month,Cos_month,is_weekend,bank_hol,school_hol
0,0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1,1
1,1,2019-01-01,Bradford - BID,54153.485,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1,1
2,2,2019-01-01,Bradford - Local Authority,530996.000,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1,1
3,3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0,0,1
4,4,2019-01-02,Bradford - BID,158891.385,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0,0,1


## Step 4: Covid
The first covid lock down occured on the 23rd of March 2020 and the last covid restrictions were lifted on the 24 February 2022 in England. Thus, dates that fall within this period will be assigned a 1 in the 'covid' column.

In [12]:
#Create set of dates during covid times in UK
start= '2020-03-23'
end= '2022-02-24'

covid_dates = pd.date_range(start=start, end=end, freq='D')

#Add column in footfall data (1= covid, 0= normal day)
footfall['covid'] = footfall['datestamp'].isin(covid_dates).astype(int)
#Check
footfall.head()

,Unnamed: 0,datestamp,area,estimated_actual_footfall,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_monthday,Cos_monthday,Sin_week_of_year,Cos_week_of_year,Sin_month,Cos_month,is_weekend,bank_hol,school_hol,covid
0,0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1,1,0
1,1,2019-01-01,Bradford - BID,54153.485,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1,1,0
2,2,2019-01-01,Bradford - Local Authority,530996.000,2019,1,1,1,0.866025,0.5,0.201299,0.979530,0.118273,0.992981,0.5,0.866025,0,1,1,0
3,3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0,0,1,0
4,4,2019-01-02,Bradford - BID,158891.385,2019,1,2,1,0.866025,-0.5,0.394356,0.918958,0.118273,0.992981,0.5,0.866025,0,0,1,0


## Step 5: Weather

This section collects daily weather data (average temperature, total precipitation, maximum wind speed) for the diffrent locations over the study period (2019-2025) using the Open Meteo API.

In [13]:
pip install --update typing extensions

Note: you may need to restart the kernel to use updated packages.



[optparse.groups]Usage:[/]   
  c:\ProgramData\anaconda3\envs\timeseries_311\python.exe -m pip install \[options] <requirement specifier> \[package-index-options] ...
  c:\ProgramData\anaconda3\envs\timeseries_311\python.exe -m pip install \[options] -r <requirements file> \[package-index-options] ...
  c:\ProgramData\anaconda3\envs\timeseries_311\python.exe -m pip install \[options] [-e] <vcs project url> ...
  c:\ProgramData\anaconda3\envs\timeseries_311\python.exe -m pip install \[options] [-e] <local project path> ...
  c:\ProgramData\anaconda3\envs\timeseries_311\python.exe -m pip install \[options] <archive url/path> ...

no such option: --update


In [14]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests

In [15]:
#Import packages
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [16]:
footfall['area'].unique()

array(['BD Walls : Wayfinders', 'Bradford - BID',
       'Bradford - Local Authority', 'BD Walls : Serving the district',
       'BD Walls : The Portal', 'Bradford - City Centre',
       'Bradford - Lister Park', 'Bradford - Roberts Park',
       'Darley Street Market', 'BD Walls : Come on in my friend',
       'BD Walls : Roots'], dtype=object)

To scrap weather data from the API, I need coordinates for each location. For the locations with polygons, the coordinates correspond to the polygon's centroid. Otherwise, the coordinates of location points is used.

**Important Note:** Here I extracted the address of the BD Walls and Darley Street Market locations from the web, to get point coordinates. However, these 'Point' geometries are only for weather scraping purposes. In reality, the footfall data was collected for the area surrounding these points (customized geometries), but I have unfortunately not been provided the geometries for these.


**Create coordinates for the BD walls and Darley Street Market:**

The adress/location of the different BD Walls was collected from the [Bradford 2025 website](https://bradford2025.co.uk/programme/bd-walls/).

* 'BD Walls : Wayfinders': 15 Park Rd, Bingley BD16 4BD, UK
* 'BD Walls : Serving the district': 275 Bradford Rd, Idle, Bradford BD10 8EG, UK
* 'BD Walls : The Portal': 8 Church Bank, Bradford BD1 4DZ, UK
* 'BD Walls : Come on in my friend' : Reevy Rd, Bradford BD6 3PU, UK
* 'BD Walls : Roots': 1 Coates St, Bradford BD5 7DL, UK
* 'Darley Street Market'

In [17]:
footfall['area'].unique()

array(['BD Walls : Wayfinders', 'Bradford - BID',
       'Bradford - Local Authority', 'BD Walls : Serving the district',
       'BD Walls : The Portal', 'Bradford - City Centre',
       'Bradford - Lister Park', 'Bradford - Roberts Park',
       'Darley Street Market', 'BD Walls : Come on in my friend',
       'BD Walls : Roots'], dtype=object)

In [18]:
#List of locations and their coordinates (name, lat, lon) or centroids

locations= [
    ('Bradford - BID', 53.79375, -1.75189),
    ('Bradford - Local Authority', 53.84471, -1.85473),
    ('Bradford - City Centre', 53.79221, -1.75405),
    ('Bradford - Lister Park', 53.81292, -1.77294),
    ('Bradford - Roberts Park', 53.81292, -1.77294),
    ('BD Walls : Wayfinders', 53.8494997,-1.8396872),
    ('BD Walls : The Portal', 53.7953434,-1.7494453),
    ('BD Walls : Serving the district', 53.8249981,-1.7367139),
    ('BD Walls : Come on in my friend', 53.7647511, -1.7903011),
    ('BD Walls : Roots', 53.7797565,-1.7605108),
    ('Darley Street Market', 53.7955833,-1.7724355)
]

In [19]:
#Code provided by Open-Meteo API website

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

#Contain waether for all locations
all_weather = []

#Collect weather data for all 4 locations
for name, lat, lon in locations:

	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	url = "https://archive-api.open-meteo.com/v1/archive"
	params = {
		"latitude": lat,
		"longitude": lon,
		"start_date": "2019-01-01",
		"end_date": "2025-12-31",
		"daily": ["temperature_2m_mean", "precipitation_sum", "wind_speed_10m_max", "daylight_duration"],
	}

	responses = openmeteo.weather_api(url, params=params)
	# Process location
	response = responses[0]


	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
	daily_precipitation_sum = daily.Variables(1).ValuesAsNumpy()
	daily_wind_speed_10m_max = daily.Variables(2).ValuesAsNumpy()
	daily_daylight_duration = daily.Variables(3).ValuesAsNumpy()

	daily_data = {"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)}

	daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
	daily_data["precipitation_sum"] = daily_precipitation_sum
	daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
	daily_data["daylight_duration"] = daily_daylight_duration

	daily_weather = pd.DataFrame(data = daily_data)
	daily_weather['region'] = name
	all_weather.append(daily_weather)
 
#Combine all regions into single dataframe
daily_weather_all = pd.concat(all_weather, ignore_index= True)
print(daily_weather_all.head())

                       date  temperature_2m_mean  precipitation_sum  \
0 2019-01-01 00:00:00+00:00             6.158333                0.0   
1 2019-01-02 00:00:00+00:00             2.304167                0.0   
2 2019-01-03 00:00:00+00:00             0.387500                0.0   
3 2019-01-04 00:00:00+00:00             2.064583                0.0   
4 2019-01-05 00:00:00+00:00             3.768750                0.0   

   wind_speed_10m_max  daylight_duration          region  
0           24.490587       27067.837891  Bradford - BID  
1           10.086427       27147.843750  Bradford - BID  
2            6.480000       27235.123047  Bradford - BID  
3           15.629971       27329.416016  Bradford - BID  
4           14.168641       27430.486328  Bradford - BID  


In [20]:
#Rename column
daily_weather_all= daily_weather_all.rename(columns= {'date': 'datestamp'})
daily_weather_all= daily_weather_all.rename(columns= {'region': 'area'})

#Convert datestamp (avoids having hour:min:secs) to timezone naive
daily_weather_all['datestamp'] = (pd.to_datetime(daily_weather_all['datestamp'])
                              .dt.tz_localize(None)
                              .dt.normalize())

In [21]:
daily_weather_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28127 entries, 0 to 28126
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   datestamp            28127 non-null  datetime64[ns]
 1   temperature_2m_mean  28127 non-null  float32       
 2   precipitation_sum    28127 non-null  float32       
 3   wind_speed_10m_max   28127 non-null  float32       
 4   daylight_duration    28127 non-null  float32       
 5   area                 28127 non-null  object        
dtypes: datetime64[ns](1), float32(4), object(1)
memory usage: 879.1+ KB


In [22]:
print(daily_weather_all.columns)
print(footfall.columns)

Index(['datestamp', 'temperature_2m_mean', 'precipitation_sum',
       'wind_speed_10m_max', 'daylight_duration', 'area'],
      dtype='object')
Index(['Unnamed: 0', 'datestamp', 'area', 'estimated_actual_footfall', 'year',
       'month', 'weekday', 'week_of_year', 'Sin_weekday', 'Cos_weekday',
       'Sin_monthday', 'Cos_monthday', 'Sin_week_of_year', 'Cos_week_of_year',
       'Sin_month', 'Cos_month', 'is_weekend', 'bank_hol', 'school_hol',
       'covid'],
      dtype='object')


In [23]:
#Merge the weather data and the footfall data based on dates and area name
footfall_clean = footfall.merge(daily_weather_all[['datestamp', 'area', 'temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'daylight_duration']], 
                                on=['datestamp', 'area'],
                                how='left')
footfall_clean.head()

,Unnamed: 0,datestamp,area,estimated_actual_footfall,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,...,Sin_month,Cos_month,is_weekend,bank_hol,school_hol,covid,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,daylight_duration
0,0,2019-01-01,BD Walls : Wayfinders,79795.860,2019,1,1,1,0.866025,0.5,...,0.5,0.866025,0,1,1,0,6.306499,0.2,27.722336,27019.876953
1,1,2019-01-01,Bradford - BID,54153.485,2019,1,1,1,0.866025,0.5,...,0.5,0.866025,0,1,1,0,6.158333,0.0,24.490587,27067.837891
2,2,2019-01-01,Bradford - Local Authority,530996.000,2019,1,1,1,0.866025,0.5,...,0.5,0.866025,0,1,1,0,6.024499,0.1,28.460497,27067.837891
3,3,2019-01-02,BD Walls : Wayfinders,17168.990,2019,1,2,1,0.866025,-0.5,...,0.5,0.866025,0,0,1,0,1.856500,0.0,8.225035,27100.183594
4,4,2019-01-02,Bradford - BID,158891.385,2019,1,2,1,0.866025,-0.5,...,0.5,0.866025,0,0,1,0,2.304167,0.0,10.086427,27147.843750


In [24]:
footfall_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27186 entries, 0 to 27185
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Unnamed: 0                 27186 non-null  int64         
 1   datestamp                  27186 non-null  datetime64[ns]
 2   area                       27186 non-null  object        
 3   estimated_actual_footfall  27186 non-null  float64       
 4   year                       27186 non-null  int64         
 5   month                      27186 non-null  int64         
 6   weekday                    27186 non-null  int64         
 7   week_of_year               27186 non-null  int64         
 8   Sin_weekday                27186 non-null  float64       
 9   Cos_weekday                27186 non-null  float64       
 10  Sin_monthday               27186 non-null  float64       
 11  Cos_monthday               27186 non-null  float64       
 12  Sin_

## Step 6: Location choice + One-hot encoding location names

**IMPORTANT NOTE**: The '2- Footfall-Cleaning' notebook is the same between the different analysis versions tested. However, the '3- Data Scaping' and '4- Modelling' notebooks are changed to only include the locations of interest for this version of the analysis.

**Version of the analysis:** only for the polygons, and thus will only include the following locations:
* Bradford - BID
* Bradford - City Centre
* Bradford - Lister Park
* Bradford - Roberts Park
* Bradford - Local Authority

In [25]:
#Only include the locations used in this version of the analysis
# Bradford - BID
# Bradford - City Centre
# Bradford - Lister Park
# Bradford - Roberts Park
# Bradford - Local Authority

footfall_clean = footfall_clean[footfall_clean['area'].isin(['Bradford - BID',
                                    'Bradford - City Centre',
                                    'Bradford - Lister Park',
                                    'Bradford - Roberts Park',
                                    'Bradford - Local Authority'])]

In [26]:
#Duplicate the area column
footfall_clean['area_name'] = footfall_clean['area']
#Convert region names to binary columns
footfall_clean = pd.get_dummies(footfall_clean, columns=['area'], dtype=int)
#Check
footfall_clean.head()

,Unnamed: 0,datestamp,estimated_actual_footfall,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_monthday,...,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,daylight_duration,area_name,area_Bradford - BID,area_Bradford - City Centre,area_Bradford - Lister Park,area_Bradford - Local Authority,area_Bradford - Roberts Park
1,1,2019-01-01,54153.485,2019,1,1,1,8.660254e-01,0.5,0.201299,...,6.158333,0.0,24.490587,27067.837891,Bradford - BID,1,0,0,0,0
2,2,2019-01-01,530996.000,2019,1,1,1,8.660254e-01,0.5,0.201299,...,6.024499,0.1,28.460497,27067.837891,Bradford - Local Authority,0,0,0,1,0
4,4,2019-01-02,158891.385,2019,1,2,1,8.660254e-01,-0.5,0.394356,...,2.304167,0.0,10.086427,27147.843750,Bradford - BID,1,0,0,0,0
5,5,2019-01-02,568621.000,2019,1,2,1,8.660254e-01,-0.5,0.394356,...,1.566167,0.0,8.496305,27147.843750,Bradford - Local Authority,0,0,0,1,0
7,7,2019-01-03,56947.585,2019,1,3,1,1.224647e-16,-1.0,0.571268,...,0.387500,0.0,6.480000,27235.123047,Bradford - BID,1,0,0,0,0


## Step 7: Format data for Modelling

The model will be using the years 2019 to 2024 for training, thus the year 2025 needs to be separated.

In [27]:
footfall_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12395 entries, 1 to 27184
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Unnamed: 0                       12395 non-null  int64         
 1   datestamp                        12395 non-null  datetime64[ns]
 2   estimated_actual_footfall        12395 non-null  float64       
 3   year                             12395 non-null  int64         
 4   month                            12395 non-null  int64         
 5   weekday                          12395 non-null  int64         
 6   week_of_year                     12395 non-null  int64         
 7   Sin_weekday                      12395 non-null  float64       
 8   Cos_weekday                      12395 non-null  float64       
 9   Sin_monthday                     12395 non-null  float64       
 10  Cos_monthday                     12395 non-null  float64       

In [28]:
#Drop unneeded columns
footfall_clean = footfall_clean.drop(columns=['Unnamed: 0'])

In [29]:
footfall_2025 = footfall_clean[footfall_clean['year'] == 2025]
footfall_19_24 = footfall_clean[footfall_clean['year'] != 2025]

In [30]:
footfall_19_24.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10862 entries, 1 to 23563
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   datestamp                        10862 non-null  datetime64[ns]
 1   estimated_actual_footfall        10862 non-null  float64       
 2   year                             10862 non-null  int64         
 3   month                            10862 non-null  int64         
 4   weekday                          10862 non-null  int64         
 5   week_of_year                     10862 non-null  int64         
 6   Sin_weekday                      10862 non-null  float64       
 7   Cos_weekday                      10862 non-null  float64       
 8   Sin_monthday                     10862 non-null  float64       
 9   Cos_monthday                     10862 non-null  float64       
 10  Sin_week_of_year                 10862 non-null  float64       

## Step 8: Outlier removal
The model should predict normal footfall. Therefore any days that have extremely high or low footfall should be taken out of the training data. We don't actually want the model to try to predict footfall on unusual days, because the things that make the day unusual (like errors, or the presence of special events) are not captured in the input data.
Dates with very high or very low footfall counts are excluded using the Median Average Distance technique.

In [31]:
#Create function to find outliers using the Median Average Distance
#The outliers are removed from the training data (2019 to 2024)

def doubleMADsfromMedian(y,thresh=3.5):
    """
    Find outliers using the Median Average Distance.
    
    VALUE: return a list of true/false denoting whether the element in y is an outlier or not
    
    PARAMETERS:
      - y is a pandas Series, or something like that.
    
    warning: this function does not check for NAs
    nor does it address issues when 
    more than 50% of your data have identical values
    """
    # Calculate the upper and lower limits
    m = np.median(y) # The median
    abs_dev = np.abs(y - m) # The absolute difference between each y and the median
    # The upper and lower limits are the median of the difference
    # of each data point from the median of the data
    left_mad = np.median(abs_dev[y <= m]) # The left limit (median of lower half)
    right_mad = np.median(abs_dev[y >= m]) # The right limit (median of upper half)
    
    # Now create an array where each value has left_mad if it is in the lower half of the data,
    # or right_mad if it is in the upper half
    y_mad = left_mad * np.ones(len(y)) # Initially every value is 'left_mad'
    y_mad[y > m] = right_mad # Now larger values are right_mad

    # Calculate the z scores for each element
    modified_z_score = 0.6745 * abs_dev / y_mad
    modified_z_score[y == m] = 0
    
    # Return boolean list showing whether each y is an outlier
    return modified_z_score > thresh

# Make a list of true/false for whether the footfall is an outlier
no_outliers = pd.DataFrame(doubleMADsfromMedian(footfall_19_24['estimated_actual_footfall']))
no_outliers.columns = ['outlier'] # Rename the column to 'outlier'

# Join to the original footfall data to the list of outliers, then select a few useful columns
join = pd.concat([footfall_19_24, no_outliers], axis = 1)
join = pd.DataFrame(join, columns = ['datestamp', 'outlier', 'estimated_actual_footfall'])

# Choose just the outliers
outliers = join[join['outlier'] == True]
outliers_idx = outliers.index

# Now remove all outliers from the original data
df = footfall_19_24.loc[~footfall_19_24.index.isin(outliers_idx)]
df = df.reset_index(drop = True)

# Check that the lengths all make sense
assert(len(df) == len(footfall_19_24)-len(outliers_idx))

print("I found {} outliers from {} days in total. Removing them leaves us with {} events".format(\
    len(outliers_idx), len(join), len(df) ) )

I found 2192 outliers from 10862 days in total. Removing them leaves us with 8670 events


That is a lot of outliers considering there is not that many data points, and this would totally take out all data points from the 'Bradford - Local Authority', so I decided to keep all data points.

In [32]:
#Reassign the new dataset (excluding outliers) to footfall_19_24
#footfall_19_24 = df.copy()

## Step 9: Save the 3 cleaned datasets, ready for analysis

In [33]:
footfall_clean.to_csv('footfall_cleaned.csv')
footfall_19_24.to_csv('footfall_cleaned_19_24.csv')
footfall_2025.to_csv('footfall_cleaned_2025.csv')

In [34]:
footfall_19_24['area_name'].unique()

array(['Bradford - BID', 'Bradford - Local Authority',
       'Bradford - City Centre', 'Bradford - Lister Park',
       'Bradford - Roberts Park'], dtype=object)